## 1 setup

In [4]:
%pip install -q "datasets==3.6.0" pandas pyarrow numpy scipy scikit-learn tqdm matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import ast #per convertire liste e diz python in oggetti veri
import json 
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display #per mostrare meglio tabelle e DataFrame
from scipy.sparse import csr_matrix #per matrici sparse
from sklearn.decomposition import TruncatedSVD #per fare decomposizione della matrice utente-prodotto
from sklearn.neighbors import NearestNeighbors #per trovare elementi simili
from tqdm.auto import tqdm

pd.set_option("display.max_columns",80)
pd.set_option("display.max_colwidth",120)

In [6]:
CATEGORY="Electronics"
RANDOM_STATE=42
K_VALUES=(5,10,20)
PRIMARY_METRIC="ndcg" 
#dobbiamo usare ndcg come metrica, perchè misura quanto bene il modello ordina gli elementi
#nelle prime k posizioni
#sommatoria per le prime k pos di (rilevanza elemento in posizione i/log2(i+1))
#se prodotto fuori dalla topk è uguale a 0
PRIMARY_K=10
#scegli che modelli seguire
RUN_POPULARITY=True
RUN_ITEMKNN=True
RUN_SVD=True

USE_RATING_VALUES=False #stimiamo se utente interagisce col prodotto, non il rating che deve dare
MAX_EVAL_ROWS=None #quante righe usare al max nella valutazione, con none le usi tutte
SAVE_RANKS=True #salva i rank prodotti dai modelli
SAVE_MODEL_ARTIFACTS=True #salva artefatti,mapping..

ITEMKNN_NEIGHBORS=50 #guarda 50 prodotti vicini/simili
ITEMKNN_N_JOBS=-1 #usa tutte le risorse disponibili

SVD_FACTORS=[32,64] #fattori latenti da provare per truncatedSVD per approssimare la matrice utenti-prodotti
SVD_N_ITER=7 #num iterazioni di svd

DEMO_USERS_N=3 #3 utenti demo
DEMO_TOP_N=10 #mostra 10 raccomandazioni per utente

## 2 percorsi e caricamento dati

In [ ]:
PROJECT_ROOT=Path("..").resolve() if (Path("..")/"data").exists() else Path(".").resolve()
RECOMMENDER_DIR=PROJECT_ROOT/"data"/"processed"/"electronics_recommender"
NOTEBOOK04_DIR=RECOMMENDER_DIR/"notebook04_models"
NOTEBOOK04_DIR.mkdir(parents=True,exist_ok=True)

TRAIN_PATH=RECOMMENDER_DIR/"train.parquet"
VALIDATION_EVAL_PATH=RECOMMENDER_DIR/"validation_eval.parquet"
TEST_EVAL_PATH=RECOMMENDER_DIR/"test_eval.parquet"
VALIDATION_NEGATIVE_PATH=RECOMMENDER_DIR/"negative_samples_validation.parquet"
TEST_NEGATIVE_PATH=RECOMMENDER_DIR/"negative_samples_test.parquet"
ITEMS_PATH=RECOMMENDER_DIR/"items.parquet"
USER_MAPPING_PATH=RECOMMENDER_DIR/"user_mapping.parquet"
ITEM_MAPPING_PATH=RECOMMENDER_DIR/"item_mapping.parquet"
SPLIT_SUMMARY_PATH=RECOMMENDER_DIR/"split_summary.json"

required_paths={
    "train":TRAIN_PATH,
    "validation_eval":VALIDATION_EVAL_PATH,
    "test_eval":TEST_EVAL_PATH,
    "negative_samples_validation":VALIDATION_NEGATIVE_PATH,
    "negative_samples_test":TEST_NEGATIVE_PATH,
    "items":ITEMS_PATH,
    "user_mapping":USER_MAPPING_PATH,
    "item_mapping":ITEM_MAPPING_PATH,
}
missing_paths=[name for name,path in required_paths.items() if not path.exists()]
print("Project root:",PROJECT_ROOT)
print("Recommender dir:",RECOMMENDER_DIR)
print("Notebook04 dir:",NOTEBOOK04_DIR)

for name,path in required_paths.items():
    print(f"{name}:{path} | exists={path.exists()}")
if missing_paths:
    raise FileNotFoundError("File mancanti dal Notebook 03:"+",".join(missing_paths))

Project root: C:\Users\arsid\Desktop\Tesi-ecommerce
Recommender dir: C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\electronics_recommender
Notebook04 dir: C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\electronics_recommender\notebook04_models
train:C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\electronics_recommender\train.parquet | exists=True
validation_eval:C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\electronics_recommender\validation_eval.parquet | exists=True
test_eval:C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\electronics_recommender\test_eval.parquet | exists=True
negative_samples_validation:C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\electronics_recommender\negative_samples_validation.parquet | exists=True
negative_samples_test:C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\electronics_recommender\negative_samples_test.parquet | exists=True
items:C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\electronics_recommender\i

In [8]:
train_df=pd.read_parquet(TRAIN_PATH)
validation_eval_df=pd.read_parquet(VALIDATION_EVAL_PATH)
test_eval_df=pd.read_parquet(TEST_EVAL_PATH)
validation_samples_df=pd.read_parquet(VALIDATION_NEGATIVE_PATH)
test_samples_df=pd.read_parquet(TEST_NEGATIVE_PATH)
items_df=pd.read_parquet(ITEMS_PATH)
user_mapping_df=pd.read_parquet(USER_MAPPING_PATH)
item_mapping_df=pd.read_parquet(ITEM_MAPPING_PATH)

if SPLIT_SUMMARY_PATH.exists():
    split_summary=json.loads(SPLIT_SUMMARY_PATH.read_text(encoding="utf-8"))
else:
    split_summary={}

loaded_shapes=pd.DataFrame([
    {"name":"train","rows":len(train_df),"cols":train_df.shape[1]},
    {"name":"validation_eval","rows":len(validation_eval_df),"cols":validation_eval_df.shape[1]},
    {"name":"test_eval","rows":len(test_eval_df),"cols":test_eval_df.shape[1]},
    {"name":"validation_samples","rows":len(validation_samples_df),"cols":validation_samples_df.shape[1]},
    {"name":"test_samples","rows":len(test_samples_df),"cols":test_samples_df.shape[1]},
    {"name":"items","rows":len(items_df),"cols":items_df.shape[1]},
    {"name":"user_mapping","rows":len(user_mapping_df),"cols":user_mapping_df.shape[1]},
    {"name":"item_mapping","rows":len(item_mapping_df),"cols":item_mapping_df.shape[1]},
])

display(loaded_shapes)
display(pd.DataFrame.from_dict(split_summary,orient="index",columns=["value"]).head(30))

,name,rows,cols
0,train,287752,14
1,validation_eval,131376,14
2,test_eval,131352,14
3,validation_samples,131376,7
4,test_samples,131352,7
5,items,12306,13
6,user_mapping,131398,2
7,item_mapping,12306,2


,value
category,Electronics
source_config,user3_item8
positive_rating_threshold,4
min_user_interactions_after,3
min_item_interactions_after,8
n_negatives,100
random_state,42
temporal_column,datetime
original_candidate_interactions,844060
positive_interactions_before_refilter,671903


In [10]:
def parse_negative_items(value): #prende valore della colonna negative_item_idxs e prova a convertirlo in lista
    if value is None:
        return []
    if isinstance(value,np.ndarray): #se array lo converte in lista python
        value=value.tolist()
    if isinstance(value,(list,tuple,set)): #se già lista, converte ogni elemento in int
        return [int(item) for item in value]
    if isinstance(value,str): #se valore è una stringa, toglie spazi inutili e prova a interpretarla come lista python
        value=value.strip()
        if not value:
            return []
        parsed=ast.literal_eval(value)
        return [int(item) for item in parsed]
    try:
        if pd.isna(value):
            return []
    except (TypeError,ValueError):
        pass
    return [int(value)]

validation_samples_df=validation_samples_df.copy()
test_samples_df=test_samples_df.copy()

validation_samples_df["negative_item_idxs"]=validation_samples_df["negative_item_idxs"].apply(parse_negative_items)
test_samples_df["negative_item_idxs"]=test_samples_df["negative_item_idxs"].apply(parse_negative_items)

if MAX_EVAL_ROWS is not None: #se non nullo, limita righe usate per valutazione
    validation_samples_df=validation_samples_df.head(int(MAX_EVAL_ROWS)).copy()
    test_samples_df=test_samples_df.head(int(MAX_EVAL_ROWS)).copy()

print("Validation samples:",len(validation_samples_df))
print("Test samples:",len(test_samples_df))
display(validation_samples_df.head())

Validation samples: 131376
Test samples: 131352


,split,user_idx,user_id,positive_item_idx,positive_parent_asin,negative_item_idxs,num_negatives
0,validation,0,AE22236AFRRSMQIKGG7TPTB75QEA,4617,B003ZDO2LM,"[259, 3414, 5268, 10203, 6251, 6543, 3981, 11457, 3205, 9702, 5687, 6513, 6868, 1934, 11272, 5301, 5897, 8066, 5849,...",100
1,validation,1,AE222H3FGXWLHRFUMGMS2RR57NDQ,6050,B0054JJ0QW,"[7710, 7535, 9359, 6955, 9631, 3850, 11767, 2880, 10843, 9619, 10135, 10058, 1190, 8283, 11116, 4856, 624, 1282, 100...",100
2,validation,2,AE223NW3QAU2KTAY5MAEU6VPBAUA,5807,B004VRQNS6,"[7396, 842, 654, 6114, 7341, 9000, 11775, 6275, 8528, 1200, 3251, 189, 2318, 637, 8829, 9776, 1865, 5393, 10232, 103...",100
3,validation,3,AE224HM2QAW5TTSDL2QRDERA6KMA,343,B0000DBJ8A,"[7445, 9637, 11790, 8567, 7608, 6313, 8444, 2044, 3338, 4990, 87, 6805, 7189, 1710, 3695, 5050, 5386, 8303, 11151, 7...",100
4,validation,4,AE2252DKW4XJIZP5QPFMQVJBVRTA,2444,B001L5SN7U,"[10098, 5608, 9372, 12247, 5178, 4924, 4000, 351, 5878, 3300, 6104, 1334, 14, 1952, 10369, 4984, 1346, 3484, 19, 182...",100


## 3 matrice utente prodotto